In [36]:
import csv
import heapq

# Define the job seeker class
class JobSeeker:
    def __init__(self, skills, experience, salary, location, job_interest, sector, education_level, job_id):
        self.skills = skills
        self.experience = experience
        self.salary = salary
        self.location = location
        self.job_interest = job_interest
        self.sector = sector
        self.education_level = education_level
        self.job_id = job_id

    def __repr__(self):
        return (f"JobSeeker(job_id={self.job_id}, skills={self.skills}, "
                f"experience={self.experience}, salary={self.salary}, "
                f"location={self.location}, job_interest={self.job_interest}, "
                f"sector={self.sector}, education_level={self.education_level})")

# Define the job offer class
class JobOffer:
    def __init__(self, required_skills, min_experience, salary_range, location, sector, education_level):
        self.required_skills = required_skills
        self.min_experience = min_experience
        self.salary_range = salary_range
        self.location = location
        self.sector = sector
        self.education_level = education_level

    def __repr__(self):
        return (f"JobOffer(required_skills={self.required_skills}, "
                f"min_experience={self.min_experience}, "
                f"salary_range={self.salary_range}, "
                f"location={self.location}, "
                f"sector={self.sector}, "
                f"education_level={self.education_level})")

# Load job seekers from CSV
def load_job_seekers(filename):
    job_seekers = []
    try:
        with open(filename, mode='r') as file:
            reader = csv.DictReader(file)
            for row in reader:
                job_seekers.append(JobSeeker(
                    skills=row['skills'].split(', '),
                    experience=int(row['experience']),
                    salary=int(row['salary']),
                    location=row['location'],
                    job_interest=row['job_interest'],
                    sector=row['sector'],
                    education_level=row['education_level'],
                    job_id=row['job_id']
                ))
        print(f"Loaded {len(job_seekers)} job seekers.")
    except FileNotFoundError:
        print(f"Error: The file {filename} was not found.")
    except Exception as e:
        print(f"Error: {e}")
    return job_seekers

# Calculate the cost of a job seeker for a specific job offer (based on mismatch)
def cost(job_seeker, job_offer):
    skill_mismatch = len(set(job_seeker.skills) - set(job_offer.required_skills))
    experience_mismatch = max(0, job_offer.min_experience - job_seeker.experience)
    salary_mismatch = max(0, job_offer.salary_range[0] - job_seeker.salary) + max(0, job_seeker.salary - job_offer.salary_range[1])
    location_mismatch = 0 if job_seeker.location == job_offer.location else 1
    return skill_mismatch + experience_mismatch + salary_mismatch + location_mismatch

# Improved heuristic to provide better matching for job seekers
def improved_heuristic(job_seeker, job_offer):
    # Skills mismatch: penalty for each missing skill
    missing_skills = set(job_offer.required_skills) - set(job_seeker.skills)
    skill_penalty = len(missing_skills) * 10  # Weight each missing skill with a penalty of 10
    
    # Experience mismatch: penalty for not having enough experience
    experience_penalty = max(0, job_offer.min_experience - job_seeker.experience) * 5  # Weight each year of missing experience with a penalty of 5
    
    # Salary mismatch: penalty if salary is outside the job offer's salary range
    salary_penalty = 0
    if job_seeker.salary < job_offer.salary_range[0]:
        salary_penalty = (job_offer.salary_range[0] - job_seeker.salary) * 2  # Weight salary mismatch with a factor of 2
    elif job_seeker.salary > job_offer.salary_range[1]:
        salary_penalty = (job_seeker.salary - job_offer.salary_range[1]) * 2  # Weight salary mismatch with a factor of 2
    
    # Location mismatch: penalty if the location doesn't match
    location_penalty = 0 if job_seeker.location == job_offer.location else 10  # 10 penalty if location doesn't match
    
    # Combine all penalties into the final heuristic value
    total_penalty = skill_penalty + experience_penalty + salary_penalty + location_penalty
    
    return total_penalty

# A* search algorithm with the improved heuristic
def a_star_search(job_seekers, job_offer):
    best_seeker = None
    best_heuristic = float('inf')  # Initialize with a large value to ensure we get a better one
    
    # Evaluate each job seeker using the heuristic
    for job_seeker in job_seekers:
        seeker_heuristic = improved_heuristic(job_seeker, job_offer)
        
        # We select the best match with the lowest heuristic value
        if seeker_heuristic < best_heuristic:
            best_heuristic = seeker_heuristic
            best_seeker = job_seeker

    return best_seeker


# Define sample job offers for testing
def define_sample_jobs():
    job1 = JobOffer(required_skills=["Copywriting"], min_experience=3, salary_range=(50000, 70000), location="Algiers", sector="Tech", education_level="Bachelor")
    job2 = JobOffer(required_skills=["Java", "Data Analysis"], min_experience=2, salary_range=(40000, 60000), location="Tizi ouzou", sector="Tech", education_level="Master")
    job3 = JobOffer(
            required_skills=["SEO", "Google Ads"],
            min_experience=3,
            salary_range=(50000, 90000),
            location="Algiers",
            sector="Business",
            education_level="Master's"
        )
    job4 = JobOffer(
            required_skills=["Java", "SQL"],
            min_experience=2,
            salary_range=(40000, 80000),
            location="Tizi Ouzou",
            sector="Information Technology",
            education_level="Bachelor's"
        )
    return [job1, job2, job3, job4]


# Main function to test the implementation
def main():
    job_seekers = load_job_seekers("job_seekers_with_ids.csv")
    
    if not job_seekers:
        print("No job seekers found. Check the file.")
        return  # Exit the function if no job seekers were loaded
    
    job_offers = define_sample_jobs()

    # Test A* search on each job offer
    for job_offer in job_offers:
        best_match = a_star_search(job_seekers, job_offer)
        if best_match:
            print(f"Best match job seeker found with ID: {best_match.job_id}")
        else:
            print("No suitable match found.")


if __name__ == "__main__":
    main()


Loaded 10000 job seekers.
Best match job seeker found with ID: 10176
Best match job seeker found with ID: 10113
Best match job seeker found with ID: 10176
Best match job seeker found with ID: 10089
